# CREM: Transacciones Inmobiliarias por Municipio
**Propósito**: Extrae la información trimestral sobre transacciones inmobiliarias por municipio a partir de la página oficial del CREM (Centro Regional de Estadística de Murcia), procesa y limpia los datos directamente en memoria y los almacena en una tabla Delta Lake en Spark.

**Particularidades de esta tabla**:
- Las columnas son trimestres (`IV Trimestre 2025`, `III Trimestre 2024`, …) en lugar de años.
- La primera fila es el total regional (MURCIA, Región de) y **no se incluye**.
- Todas las filas restantes son municipios (no hay distritos ni secciones).

In [1]:
import sys
import re
from bs4 import BeautifulSoup
import pandas as pd
from pyspark.sql import functions as F

sys.path.insert(0, '/tfm/python_notebooks')
from tfm_lib import utils as tfm_utils
from tfm_lib import crem_scrap as crem

In [2]:
# Definir la url de la web
url_base = "https://econet.carm.es/web/crem/inicio/-/crem/sicrem/PM_actNotarial/sec5_c1.html"
table_name = "raw.crem.transacciones_inmobiliarias"
column_suffix = "transacciones_inmobiliarias"

## Descarga y Parseo del Contenido HTML directamente en Memoria
Se realiza una petición HTTP para descargar el contenido HTML de la página oficial del CREM en memoria y procesarlo dinámicamente sin persistir archivos intermedios en el disco local.

In [3]:
# Hacer la solicitud
html = crem.get_html(url_base)

# Convertir HTML a BeautifulSoup object.
soup = BeautifulSoup(html, "html.parser")

table = soup.find("table", id="interiorTablaDatos")

if not table:
    raise Exception("No se encontró la tabla con id 'interiorTablaDatos' en el HTML")

# ---------------------------------------------------------------------------
# Cabeceras: la tabla tiene dos filas de thead:
#   Fila 1: trimestre ("IV", "III", "II", "I", …)
#   Fila 2: año      ("2025", "2025", …)
# Las combinamos para obtener etiquetas como "IV_Trimestre_2025".
# ---------------------------------------------------------------------------
thead = table.find("thead", id="cabeceraTablaDatos")
if not thead:
    # Algunas páginas usan <thead> sin id; fallback.
    thead = table.find("thead")

header_rows = thead.find_all("tr")

if len(header_rows) >= 2:
    # Extrae las celdas de ambas filas
    trimestres = [th.get_text(strip=True) for th in header_rows[0].find_all("th") if th.get_text(strip=True)]
    anios      = [th.get_text(strip=True) for th in header_rows[1].find_all("th") if th.get_text(strip=True)]
    # Genera columnas compuestas: "IV_Trimestre_2025", "III_Trimestre_2025", …
    trimestres_columnas = [f"{t}_Trimestre_{a}" for t, a in zip(trimestres, anios)]
else:
    # Cabecera plana (1 sola fila): tomar los th que contengan texto
    trimestres_columnas = [th.get_text(" ", strip=True) for th in header_rows[0].find_all("th") if th.get_text(strip=True)]

print(f"Columnas detectadas ({len(trimestres_columnas)}): {trimestres_columnas}")

Columnas detectadas (88): ['IV Trimestre 2025', 'III Trimestre 2025', 'II Trimestre 2025', 'I Trimestre 2025', 'IV Trimestre 2024', 'III Trimestre 2024', 'II Trimestre 2024', 'I Trimestre 2024', 'IV Trimestre 2023', 'III Trimestre 2023', 'II Trimestre 2023', 'I Trimestre 2023', 'IV Trimestre 2022', 'III Trimestre 2022', 'II Trimestre 2022', 'I Trimestre 2022', 'IV Trimestre 2021', 'III Trimestre 2021', 'II Trimestre 2021', 'I Trimestre 2021', 'IV Trimestre 2020', 'III Trimestre 2020', 'II Trimestre 2020', 'I Trimestre 2020', 'IV Trimestre 2019', 'III Trimestre 2019', 'II Trimestre 2019', 'I Trimestre 2019', 'IV Trimestre 2018', 'III Trimestre 2018', 'II Trimestre 2018', 'I Trimestre 2018', 'IV Trimestre 2017', 'III Trimestre 2017', 'II Trimestre 2017', 'I Trimestre 2017', 'IV Trimestre 2016', 'III Trimestre 2016', 'II Trimestre 2016', 'I Trimestre 2016', 'IV Trimestre 2015', 'III Trimestre 2015', 'II Trimestre 2015', 'I Trimestre 2015', 'IV Trimestre 2014', 'III Trimestre 2014', 'II Tr

In [4]:
# ---------------------------------------------------------------------------
# Parsear filas de municipios.
# La PRIMERA fila del tbody corresponde a MURCIA (Región de) → se descarta.
# El resto son municipios; en esta tabla no hay jerarquía de distritos/secciones.
# ---------------------------------------------------------------------------
tbody = table.find("tbody")
datos_filas = []
primera_fila_omitida = False

for tr in tbody.find_all("tr"):
    th = tr.find("th", id="r")
    if not th:
        continue

    # Saltar la primera fila (MURCIA, Región de)
    if not primera_fila_omitida:
        primera_fila_omitida = True
        continue

    # Nombre del municipio
    label_raw   = th.get_text(strip=True)
    label_clean = crem.clean_and_decode(label_raw)

    # Normalizar nombres del tipo "Alcázares (Los)" → "los_alcazares"
    # Primero extraemos solo el nombre (sin código numérico si lo hubiera)
    match = re.match(r"^(.*?)\s*-\s*(\d+)$", label_clean)
    nombre = match.group(1).strip() if match else label_clean

    # Mover artículos pospuestos: "Alcázares (Los)" → "Los Alcázares"
    nombre = re.sub(r"^(.+?)\s*\(([^)]+)\)$", lambda m: f"{m.group(2)} {m.group(1)}", nombre).strip()

    # Extraer valores numéricos de las celdas de datos
    tds    = tr.find_all("td", id="d", class_="tbdato")
    values = []
    for td in tds:
        val_str = td.get_text(strip=True).replace(",", ".")
        try:
            val_float = float(val_str) if val_str not in ("", "-", "..") else None
        except ValueError:
            val_float = None
        values.append(val_float)

    if len(values) == len(trimestres_columnas):
        registro = {"nombre": crem.normalize_name(nombre)}
        for col, val in zip(trimestres_columnas, values):
            registro[col] = val
        datos_filas.append(registro)

print(f"Municipios parseados con éxito: {len(datos_filas)}")
if datos_filas:
    print(f"Ejemplo: {datos_filas[0]}")

Municipios parseados con éxito: 45
Ejemplo: {'nombre': 'abanilla', 'IV Trimestre 2025': 19.0, 'III Trimestre 2025': 29.0, 'II Trimestre 2025': 25.0, 'I Trimestre 2025': 16.0, 'IV Trimestre 2024': 35.0, 'III Trimestre 2024': 18.0, 'II Trimestre 2024': 19.0, 'I Trimestre 2024': 21.0, 'IV Trimestre 2023': 14.0, 'III Trimestre 2023': 14.0, 'II Trimestre 2023': 19.0, 'I Trimestre 2023': 21.0, 'IV Trimestre 2022': 9.0, 'III Trimestre 2022': 9.0, 'II Trimestre 2022': 12.0, 'I Trimestre 2022': 16.0, 'IV Trimestre 2021': 9.0, 'III Trimestre 2021': 13.0, 'II Trimestre 2021': 23.0, 'I Trimestre 2021': 6.0, 'IV Trimestre 2020': 9.0, 'III Trimestre 2020': 6.0, 'II Trimestre 2020': 6.0, 'I Trimestre 2020': 8.0, 'IV Trimestre 2019': 15.0, 'III Trimestre 2019': 7.0, 'II Trimestre 2019': 10.0, 'I Trimestre 2019': 8.0, 'IV Trimestre 2018': 11.0, 'III Trimestre 2018': 6.0, 'II Trimestre 2018': 14.0, 'I Trimestre 2018': 15.0, 'IV Trimestre 2017': 7.0, 'III Trimestre 2017': 7.0, 'II Trimestre 2017': 6.0, '

## Integración con PySpark y Delta Lake
Se crea un Spark Session, se inicializa el Spark DataFrame a partir del listado en memoria, se normaliza su esquema con las utilidades de TFM y se guarda como Delta Table en la ruta final.

In [5]:
# Inicializar sesión de Spark usando las utilidades de TFM
spark = tfm_utils.get_spark_session(app_name="CREM_Transacciones_Inmobiliarias")

# Crear el Spark DataFrame a partir de la lista de registros estructurados
df = spark.createDataFrame(datos_filas)

# Renombrar columnas: "IV_Trimestre_2025" → "IV_Trimestre_2025_transacciones_inmobiliarias"
for col in trimestres_columnas:
    df = df.withColumnRenamed(col, f"{col}_{column_suffix}")

# Normalizar las columnas del DataFrame de acuerdo al estilo del TFM (minúsculas, sin acentos)
df_normalized = tfm_utils.normalize_df(df)

# Mostrar los primeros registros de forma estructurada
tfm_utils.display(df_normalized)

,i_trimestre_2004_transacciones_inmobiliarias,i_trimestre_2005_transacciones_inmobiliarias,i_trimestre_2006_transacciones_inmobiliarias,i_trimestre_2007_transacciones_inmobiliarias,i_trimestre_2008_transacciones_inmobiliarias,i_trimestre_2009_transacciones_inmobiliarias,i_trimestre_2010_transacciones_inmobiliarias,i_trimestre_2011_transacciones_inmobiliarias,i_trimestre_2012_transacciones_inmobiliarias,i_trimestre_2013_transacciones_inmobiliarias,...,iv_trimestre_2017_transacciones_inmobiliarias,iv_trimestre_2018_transacciones_inmobiliarias,iv_trimestre_2019_transacciones_inmobiliarias,iv_trimestre_2020_transacciones_inmobiliarias,iv_trimestre_2021_transacciones_inmobiliarias,iv_trimestre_2022_transacciones_inmobiliarias,iv_trimestre_2023_transacciones_inmobiliarias,iv_trimestre_2024_transacciones_inmobiliarias,iv_trimestre_2025_transacciones_inmobiliarias,nombre
0,18.0,16.0,16.0,32.0,30.0,24.0,16.0,4.0,6.0,8.0,...,7.0,11.0,15.0,9.0,9.0,9.0,14.0,35.0,19.0,abanilla
1,86.0,44.0,61.0,54.0,41.0,32.0,21.0,13.0,9.0,7.0,...,21.0,22.0,23.0,25.0,20.0,49.0,27.0,39.0,54.0,abaran
2,136.0,101.0,330.0,304.0,226.0,155.0,129.0,116.0,73.0,164.0,...,128.0,127.0,192.0,262.0,226.0,211.0,284.0,251.0,244.0,aguilas
3,3.0,1.0,1.0,3.0,0.0,0.0,2.0,0.0,2.0,0.0,...,1.0,3.0,6.0,0.0,3.0,2.0,5.0,3.0,4.0,albudeite
4,198.0,308.0,249.0,231.0,287.0,97.0,120.0,50.0,30.0,25.0,...,73.0,66.0,86.0,90.0,123.0,147.0,169.0,183.0,175.0,alcantarilla
5,383.0,115.0,237.0,133.0,88.0,57.0,51.0,46.0,58.0,58.0,...,98.0,146.0,121.0,128.0,301.0,196.0,226.0,317.0,316.0,los_alcazares
6,1.0,1.0,2.0,2.0,0.0,1.0,3.0,1.0,1.0,0.0,...,0.0,2.0,0.0,2.0,7.0,2.0,3.0,3.0,0.0,aledo
7,32.0,44.0,60.0,48.0,51.0,14.0,29.0,9.0,8.0,5.0,...,10.0,21.0,27.0,23.0,38.0,24.0,84.0,55.0,44.0,alguazas
8,117.0,74.0,225.0,156.0,87.0,455.0,177.0,67.0,56.0,47.0,...,110.0,87.0,90.0,130.0,128.0,149.0,124.0,142.0,186.0,alhama_de_murcia
9,68.0,87.0,114.0,175.0,91.0,85.0,28.0,20.0,35.0,11.0,...,44.0,59.0,61.0,41.0,71.0,58.0,73.0,115.0,101.0,archena


In [6]:
# Ruta destino de la Delta Table
delta_path = tfm_utils.full_table_path(table_name)

print(f"Escribiendo {df_normalized.count()} registros en la Delta Table: {table_name}")
print(f"Destino: {delta_path}")

# Escritura en modo overwrite
(df_normalized
    .write
    .mode("overwrite")
    .format("delta")
    .save(delta_path)
)

print("¡Escritura en Delta Lake completada con éxito!")

Escribiendo 45 registros en la Delta Table: raw.crem.transacciones_inmobiliarias
Destino: /tfm/delta_lake/raw/crem/transacciones_inmobiliarias
¡Escritura en Delta Lake completada con éxito!
